In [ ]:
#pip install pandas sqlalchemy psycopg2-binary python-dotenv 

In [1]:
import pandas as pd
import os

from dotenv import load_dotenv
from sqlalchemy import create_engine, URL, text

In [2]:

load_dotenv(override=True)

DB_HOST = os.getenv("DB_HOST")
DB_PORT = int(os.getenv("DB_PORT"))
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

In [3]:
connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME
)

engine = create_engine(connection_url)

In [4]:
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT version(), current_database()")
    )

    print(result.fetchone())

('PostgreSQL 16.16 (Debian 16.16-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit', 'verdejar')


In [ ]:
tables = {
    "clientes": "schema",
     "pedidos": "schema"      
}

In [ ]:
dataframes = {}

for table, schema in tables.items():

    query = f"""
        SELECT *
        FROM {schema}.{table}
    """

    dataframes[table] = pd.read_sql_query(
        query,
        engine
    )

In [ ]:
df_clientes = dataframes["clientes"]
#df_pedidos = dataframes["pedidos"]

In [ ]:
df_clientes

teste


In [ ]:
pd.read_sql(text("SELECT datname FROM pg_database WHERE NOT datistemplate;"), engine)

In [43]:
pd.read_sql(text("""
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
    ORDER BY table_schema, table_name;
"""), engine)

,table_schema,table_name


In [1]:
import requests
import pandas as pd

In [2]:
api_url = "http://127.0.0.1:8000"

def buscar_pagina(endpoint: str, page: int, page_size: int) -> dict:
    resp = requests.get(
        f"{api_url}/{endpoint}",
        params={"page": page, "page_size": page_size},
        timeout=10,
    )
    resp.raise_for_status()
    return resp.json()

In [3]:
def extrair_paginado(endpoint: str, page_size: int = 200) -> list[dict]:
    primeira = buscar_pagina(endpoint, 1, page_size)
    itens = list(primeira["items"])
    total_paginas = primeira["total_pages"]
    for page in range(2, total_paginas + 1):
        pagina = buscar_pagina(endpoint, page, page_size)
        itens.extend(pagina["items"])
    return itens

In [4]:
clientes = extrair_paginado("clientes")    
df_clientes = pd.DataFrame(clientes)       
df_clientes.shape                          

(1000, 6)

In [5]:
df_clientes

,id,nome,email,cidade,estado,cadastro_em
0,1,Diego Fernandes,diego.fernandes1@exemplo.com,Campinas,SP,2025-04-04
1,2,Zeca Ribeiro,zeca.ribeiro2@exemplo.com,Belo Horizonte,mg,2025-02-03
2,3,Diego Rocha,diego.rocha3@exemplo.com,Porto Alegre,RS,2024-06-22
3,4,Marina Lima,marina.lima4@exemplo.com,Curitiba,PR,2024-07-26
4,5,Ximena Fernandes,ximena.fernandes5@exemplo.com,Caxias do Sul,RS,2025-03-15
...,...,...,...,...,...,...
995,996,Rafael Oliveira,rafael.oliveira996@exemplo.com,Curitiba,PR,2024-09-03
996,997,Olivia Gomes,olivia.gomes997@exemplo.com,Florianopolis,SC,2024-05-03
997,998,Sofia Costa,sofia.costa998@exemplo.com,Joinville,SC,2025-03-25
998,999,Ana Souza,ana.souza999@exemplo.com,Joinville,SC,2025-08-07


In [68]:
pedidos = extrair_paginado("pedidos")
len(pedidos)    # esperado: 5100

5100

In [69]:
df_pedidos = pd.DataFrame(pedidos)       
df_pedidos.shape                

(5100, 6)

In [70]:
df_pedidos

,id,cliente_id,data_pedido,status,valor_total,itens
0,1,303,2026-02-19,entregue,"125,84","[{'produto_id': 2, 'quantidade': 3, 'preco_uni..."
1,2,853,2026-08-21,entregue,94.34,"[{'produto_id': 6, 'quantidade': 4, 'preco_uni..."
2,3,358,2026-08-25,entregue,63.76,"[{'produto_id': 4, 'quantidade': 6, 'preco_uni..."
3,4,482,2026-05-21,entregue,"90,37","[{'produto_id': 2, 'quantidade': 4, 'preco_uni..."
4,5,728,2026-10-21,entregue,"9,52","[{'produto_id': 8, 'quantidade': 2, 'preco_uni..."
...,...,...,...,...,...,...
5095,4996,735,2026-04-03,entregue,31.5,"[{'produto_id': 4, 'quantidade': 6, 'preco_uni..."
5096,4997,909,2026-05-09,cancelado,68.32,"[{'produto_id': 11, 'quantidade': 6, 'preco_un..."
5097,4998,623,16/06/2026,entregue,"60,44","[{'produto_id': 4, 'quantidade': 5, 'preco_uni..."
5098,4999,390,07/02/2026,em_transporte,"9,92","[{'produto_id': 10, 'quantidade': 2, 'preco_un..."


In [ ]:
pedidos_unicos = df_pedidos.drop_duplicates(subset="id", keep="first")

df_itens = pd.json_normalize(
    pedidos_unicos.to_dict("records"), record_path="itens", meta=["id"]
).rename(columns={"id": "pedido_id"})

df_itens["linha"] = df_itens.groupby("pedido_id").cumcount() + 1

In [ ]:
df_itens["linha"] = (
    df_itens
    .groupby("pedido_id")
    .cumcount() + 1
)

In [74]:
df_itens

,produto_id,quantidade,preco_unitario,pedido_id,linha
0,2,3,4.30,1,1
1,9,5,15.86,1,2
2,7,4,9.66,1,3
3,6,4,7.34,2,1
4,7,3,10.40,2,2
...,...,...,...,...,...
12450,5,4,6.81,4997,4
12451,4,5,5.38,4998,1
12452,12,3,11.18,4998,2
12453,10,2,4.96,4999,1


In [75]:
pedidos_unicos

,id,cliente_id,data_pedido,status,valor_total,itens
0,1,303,2026-02-19,entregue,"125,84","[{'produto_id': 2, 'quantidade': 3, 'preco_uni..."
1,2,853,2026-08-21,entregue,94.34,"[{'produto_id': 6, 'quantidade': 4, 'preco_uni..."
2,3,358,2026-08-25,entregue,63.76,"[{'produto_id': 4, 'quantidade': 6, 'preco_uni..."
3,4,482,2026-05-21,entregue,"90,37","[{'produto_id': 2, 'quantidade': 4, 'preco_uni..."
4,5,728,2026-10-21,entregue,"9,52","[{'produto_id': 8, 'quantidade': 2, 'preco_uni..."
...,...,...,...,...,...,...
5095,4996,735,2026-04-03,entregue,31.5,"[{'produto_id': 4, 'quantidade': 6, 'preco_uni..."
5096,4997,909,2026-05-09,cancelado,68.32,"[{'produto_id': 11, 'quantidade': 6, 'preco_un..."
5097,4998,623,16/06/2026,entregue,"60,44","[{'produto_id': 4, 'quantidade': 5, 'preco_uni..."
5098,4999,390,07/02/2026,em_transporte,"9,92","[{'produto_id': 10, 'quantidade': 2, 'preco_un..."
